<a href="https://colab.research.google.com/github/Angel-ag-1/ML-pipeline/blob/main/work/notebooks/w05_model_By_Angel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane


This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Angel-ag-1/ML-pipeline.git"
REPO_DIR = "ML-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/ML-pipeline/ML-pipeline/ML-pipeline


In [25]:
!pip -q install duckdb pandas pyarrow huggingface_hub scikit-learn

In [26]:
import duckdb

from huggingface_hub import login, whoami, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

login(token=HF_TOKEN)
print("Hugging Face account:", whoami()["name"])

Hugging Face account: angelhi


In [27]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

client_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
)

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
)

clients = pd.read_parquet(client_path)
march = pd.read_parquet(march_path)

print("Clients:", clients.shape)
print("March:", march.shape)

Clients: (104, 9)
March: (9841378, 30)


In [28]:
march_agg = (
    march
    .groupby("content_hash_id")
    .agg(
        client_hash_id=("client_hash_id", "first"),
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
        ga4_pageviews=("ga4_pageviews", "sum"),
    )
    .reset_index()
)


march_agg["needs_attention"] = (
    march_agg["gsc_impressions"] == 0
).astype(int)


march_agg["ctr"] = (
    march_agg["gsc_clicks"]
    / march_agg["gsc_impressions"].replace(0, pd.NA)
)

march_agg["ctr"] = march_agg["ctr"].fillna(0)

print()
print("=" * 70)
print("W05 DATASET READY")
print("=" * 70)

print("Rows:", len(march_agg))
print("Unique webpages:", march_agg["content_hash_id"].nunique())
print("Unique clients:", march_agg["client_hash_id"].nunique())

print()
print("Target distribution:")
print(march_agg["needs_attention"].value_counts())

print()
print("Missing values:")
print(
    march_agg[
        [
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "ga4_pageviews"
        ]
    ].isna().sum()
)

print()
print("First 5 rows:")
print(march_agg.head())

/tmp/ipykernel_3483/1352744671.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  march_agg["ctr"] = march_agg["ctr"].fillna(0)



W05 DATASET READY
Rows: 331437
Unique webpages: 331437
Unique clients: 55

Target distribution:
needs_attention
0    176738
1    154699
Name: count, dtype: int64

Missing values:
gsc_clicks               0
gsc_avg_position    154699
ga4_sessions             0
ga4_pageviews            0
dtype: int64

First 5 rows:
            content_hash_id           client_hash_id  gsc_impressions  \
0  content_000005d4ced12088  client_9958f0a7ae1df715               86   
1  content_00001e488b74b799  client_625b6439094e23e4                0   
2  content_00007bd2985b77c3  client_73cda7b4e4f265ea               47   
3  content_00008950670cb6b5  client_def0955f7a377868                0   
4  content_0000a348850eb1fc  client_3ffa76342f366962                0   

   gsc_clicks  gsc_avg_position  ga4_sessions  ga4_pageviews  needs_attention  \
0           0         72.854861           0.0            0.0                0   
1           0               NaN           0.0            0.0                1   
2 

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a Decision Tree because the week 4 baseline is a simple rule based prioritisation system, so a small tree provides a natural next step without  adding unnecessary complexity.

A Decision Tree can identify simple relationships between search clicks, average search position, and Google Analytics activity. It is also easier to inspect than a more complex ensemble model.

In [29]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("METHOD CHOICE")
print("=" * 70)

print("Model: Decision Tree Classifier")
print()
print("Why:")
print("- Simple and interpretable.")
print("- Suitable for a binary attention-prioritisation target.")
print("- Can capture threshold-like relationships.")
print("- Avoids unnecessary model complexity.")
print()
print("Target: needs_attention")
print("Target definition: March impressions == 0")
print()
print("Model features:")
model_features = [
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_pageviews"
]

for feature in model_features:
    print(" -", feature)

print()
print("Excluded from model:")
print(" - gsc_impressions: directly defines the target")
print(" - client_hash_id: used only for grouped validation")
print(" - content_hash_id: identifier, not a predictive feature")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


METHOD CHOICE
Model: Decision Tree Classifier

Why:
- Simple and interpretable.
- Suitable for a binary attention-prioritisation target.
- Can capture threshold-like relationships.
- Avoids unnecessary model complexity.

Target: needs_attention
Target definition: March impressions == 0

Model features:
 - gsc_clicks
 - gsc_avg_position
 - ga4_sessions
 - ga4_pageviews

Excluded from model:
 - gsc_impressions: directly defines the target
 - client_hash_id: used only for grouped validation
 - content_hash_id: identifier, not a predictive feature


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
/test
I use a grouped train split by `client_hash_id`.

This is more honest than randomly splitting individual webpages because webpages belonging to the same client can share similar traffic patterns and measurement conditions.

Using client groups means a client appears in either the training set or the test set, rather than appearing in both.

The split uses March 2026 only. No future month is used.


In [30]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.model_selection import GroupShuffleSplit

print("=" * 70)
print("GROUPED TRAIN / TEST SPLIT")
print("=" * 70)

X = march_agg[model_features].copy()
y = march_agg["needs_attention"].copy()
groups = march_agg["client_hash_id"].copy()


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Training clients:",
    groups_train.nunique()
)

print(
    "Testing clients:",
    groups_test.nunique()
)

overlap = set(groups_train) & set(groups_test)

print(
    "Client overlap between train/test:",
    len(overlap)
)

if len(overlap) == 0:
    print("✓ Group split verified: no client appears in both sets.")
else:
    print("⚠ WARNING: client overlap detected.")

print("=" * 70)
print("MISSING VALUE HANDLING")
print("=" * 70)


train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Training medians:")
print(train_medians)

print("\nRemaining missing values in training:")
print(X_train.isna().sum())

print("\nRemaining missing values in testing:")
print(X_test.isna().sum())
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


GROUPED TRAIN / TEST SPLIT
Training rows: 297082
Testing rows: 34355
Training clients: 41
Testing clients: 14
Client overlap between train/test: 0
✓ Group split verified: no client appears in both sets.
MISSING VALUE HANDLING
Training medians:
gsc_clicks          0.000000
gsc_avg_position    8.493526
ga4_sessions        0.000000
ga4_pageviews       0.000000
dtype: float64

Remaining missing values in training:
gsc_clicks          0
gsc_avg_position    0
ga4_sessions        0
ga4_pageviews       0
dtype: int64

Remaining missing values in testing:
gsc_clicks          0
gsc_avg_position    0
ga4_sessions        0
ga4_pageviews       0
dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Decision Tree is trained using only March 2026 signals that do not directly define the target.

The Week-4 baseline is calculated using the same March test rows.

The comparison uses:

- Accuracy
- ROC-AUC

Accuracy shows the percentage of correct classifications.

ROC-AUC measures how well the method ranks positive cases above negative cases across thresholds.

The same test set is used for both methods so the comparison is fair.

The baseline remains a simple rule rather than a machine learning model.

In [31]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score
)

print("=" * 70)
print("TRAINING DECISION TREE")
print("=" * 70)

model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)


model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

model_accuracy = accuracy_score(
    y_test,
    model_pred
)

model_auc = roc_auc_score(
    y_test,
    model_prob
)

model_precision = precision_score(
    y_test,
    model_pred,
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_pred,
    zero_division=0
)

print("Decision Tree trained successfully.")

print("\nDecision Tree metrics:")
print(f"Accuracy : {model_accuracy:.3f}")
print(f"ROC-AUC  : {model_auc:.3f}")
print(f"Precision: {model_precision:.3f}")
print(f"Recall   : {model_recall:.3f}")

print("=" * 70)
print("W04 BASELINE ON SAME TEST SET")
print("=" * 70)


baseline_test = march_agg.iloc[test_idx].copy()

baseline_test["baseline_score"] = (
    (baseline_test["gsc_impressions"] == 0).astype(int) * 3
    +
    (baseline_test["gsc_clicks"] == 0).astype(int) * 2
    +
    (
        baseline_test["gsc_avg_position"].fillna(100) > 20
    ).astype(int)
)


baseline_pred = (
    baseline_test["baseline_score"] >= 3
).astype(int)


baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

print(f"Accuracy : {baseline_accuracy:.3f}")
print(f"ROC-AUC  : {baseline_auc:.3f}")
print(f"Precision: {baseline_precision:.3f}")
print(f"Recall   : {baseline_recall:.3f}")

print("=" * 70)
print("MODEL VS BASELINE")
print("=" * 70)

comparison = pd.DataFrame({
    "Method": [
        "W04 Baseline",
        "Decision Tree"
    ],
    "Accuracy": [
        baseline_accuracy,
        model_accuracy
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ],
    "Recall": [
        baseline_recall,
        model_recall
    ]
})

print(comparison.to_string(index=False))

print()

if model_auc > baseline_auc:
    print("Decision Tree ROC-AUC is higher than the W04 baseline.")
elif model_auc < baseline_auc:
    print("Decision Tree ROC-AUC is lower than the W04 baseline.")
else:
    print("Decision Tree ROC-AUC is equal to the W04 baseline.")

print()
print("Important: complexity is not treated as an improvement by itself.")
print("The model must provide better measured performance to beat the baseline.")

print("=" * 70)
print("DECISION TREE FEATURE IMPORTANCE")
print("=" * 70)

importance = pd.DataFrame({
    "feature": model_features,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance.to_string(index=False))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


TRAINING DECISION TREE
Decision Tree trained successfully.

Decision Tree metrics:
Accuracy : 1.000
ROC-AUC  : 1.000
Precision: 1.000
Recall   : 1.000
W04 BASELINE ON SAME TEST SET
Accuracy : 0.885
ROC-AUC  : 1.000
Precision: 0.785
Recall   : 1.000
MODEL VS BASELINE
       Method  Accuracy  ROC_AUC  Precision  Recall
 W04 Baseline  0.884646      1.0   0.785145     1.0
Decision Tree  1.000000      1.0   1.000000     1.0

Decision Tree ROC-AUC is equal to the W04 baseline.

Important: complexity is not treated as an improvement by itself.
The model must provide better measured performance to beat the baseline.
DECISION TREE FEATURE IMPORTANCE
         feature   importance
gsc_avg_position 1.000000e+00
    ga4_sessions 1.200411e-11
      gsc_clicks 0.000000e+00
   ga4_pageviews 0.000000e+00


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I inspected the cases where the Decision Tree disagreed with the target.

False positives are webpages that the model flagged as needing attention but did not have zero impressions. False negatives are webpages with zero impressions that the model failed to flag.

The target is a simple operational proxy based on zero impressions. A webpage can have zero impressions for legitimate reasons, such as being newly published, temporarily unindexed, or having insufficient search activity.

The model also does not have access to content quality, publication date, indexing status, or other contextual information that could explain why a page has low visibility.

The feature importance results show which available March signals the tree relied on most. These should be treated as directional evidence rather than causal explanations.

The model should therefore be treated as decision support rather than an automatic content-action system.

In [32]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("ERROR ANALYSIS")
print("=" * 70)

error_analysis = X_test.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = model_pred
error_analysis["prediction_probability"] = model_prob


false_positives = error_analysis[
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 1)
]


false_negatives = error_analysis[
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print()
print("=" * 70)
print("EXAMPLE FALSE POSITIVES")
print("=" * 70)

print(
    false_positives.head(10).to_string(index=False)
)

print()
print("=" * 70)
print("EXAMPLE FALSE NEGATIVES")
print("=" * 70)

print(
    false_negatives.head(10).to_string(index=False)
)

print()
print("=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

importance = pd.DataFrame({
    "feature": model_features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance.to_string(index=False))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


ERROR ANALYSIS
False positives: 0
False negatives: 0

EXAMPLE FALSE POSITIVES
Empty DataFrame
Columns: [gsc_clicks, gsc_avg_position, ga4_sessions, ga4_pageviews, actual, predicted, prediction_probability]
Index: []

EXAMPLE FALSE NEGATIVES
Empty DataFrame
Columns: [gsc_clicks, gsc_avg_position, ga4_sessions, ga4_pageviews, actual, predicted, prediction_probability]
Index: []

FEATURE IMPORTANCE
         feature   importance
gsc_avg_position 1.000000e+00
    ga4_sessions 1.200411e-11
      gsc_clicks 0.000000e+00
   ga4_pageviews 0.000000e+00
